# Introduction to Deep Learning '26 Assignment A0

Paul Tielens, s3612031

In [ ]:
!uv pip install "umap-learn[plot]"
!uv pip install matplotlib scikit-learn

error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment
error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment


In [1]:
import torch
import numpy as np

# task 1.2
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt

print(torch.cuda.is_available()) 
print(torch.cuda.device_count()) 
print(torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'torch'

In [4]:
# constants

nr_digits = 10

data_folder = "data/"

In [5]:
# Load data
train_in = np.loadtxt(data_folder + "train_in.csv", delimiter=",")
train_out = np.loadtxt(data_folder + "train_out.csv", delimiter=",")

test_out = np.loadtxt(data_folder + "test_out.csv", delimiter=",")
test_in = np.loadtxt(data_folder + "test_in.csv", delimiter=",")


In [9]:
values, counts = np.unique(test_out, return_counts=True)
for value, count in zip(values, counts):
    print(f"{int(value)}: {count}")

print()
values, counts = np.unique(train_out, return_counts=True)
for value, count in zip(values, counts):
    print(f"{int(value)}: {count}")


# train and test data is a bit unbalanced

0: 224
1: 121
2: 101
3: 79
4: 86
5: 55
6: 90
7: 64
8: 92
9: 88

0: 319
1: 252
2: 202
3: 131
4: 122
5: 88
6: 151
7: 166
8: 144
9: 132


# Task 1.1

We would expect that digits with similar average vectors would be the most difficult to separate

In [7]:
# Calculate average vector for every digit from the training data

avg_vectors = []

for i in range(nr_digits):
    indices = np.where(train_out == i)

    vectors_i = train_in[indices]
    avg_vector_i = np.mean(vectors_i, axis=0)

    avg_vectors.append(avg_vector_i)

avg_vectors = np.array(avg_vectors)

In [8]:
# Classify the digits based on the distance between the average vector and the test vector

predictions = []

for test_in_element in test_in:
    distances = []
    for i in range(nr_digits):
        dist = np.linalg.norm(avg_vectors[i] - test_in_element)
        distances.append(dist)

    # Take the index of the digit with the lowest 'distance'
    predictions.append(np.argmin(distances))

vector_avg_accuracy = (test_out == predictions).mean() 

print(vector_avg_accuracy)

0.804


# Task 1.2: dimensionality reduction

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt

In [ ]:
train_in.shape

In [ ]:
pca = PCA(n_components=2, random_state=42)
train_pca = pca.fit_transform(train_in)
avg_train_pca = pca.transform(avg_vectors)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(train_pca[:, 0], train_pca[:, 1], c=train_out, cmap="tab10")
plt.scatter(avg_train_pca[:, 0], avg_train_pca[:, 1], c="r", marker="X")

for n in range(nr_digits):
    plt.annotate(str(n), (avg_train_pca[n, 0], avg_train_pca[n, 1]), fontsize=14, fontweight='bold')

plt.legend(*scatter.legend_elements(), title="Digit", bbox_to_anchor=(1, 1), loc='upper left')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA of train_in, colored per digit compared with average digit vectors")
plt.show()



In [ ]:
combined_data = np.vstack([train_in, avg_vectors])

tsne = TSNE(n_components=2, random_state=42, n_jobs=-1)
combined_tsne = tsne.fit_transform(combined_data)

train_tsne = combined_tsne[:len(train_in)]
avg_tsne = combined_tsne[len(train_in):]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(train_tsne[:, 0], train_tsne[:, 1], c=train_out, cmap="tab10")
plt.scatter(avg_train_tsne[:, 0], avg_train_tsne[:, 1], c="r", marker="X")

for n in range(nr_digits):
    plt.annotate(str(n), (avg_train_tsne[n, 0], avg_train_tsne[n, 1]), fontsize=14, fontweight='bold')

plt.legend(*scatter.legend_elements(), title="Digit", bbox_to_anchor=(1, 1), loc='upper left')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D TSNE of train_in, colored per digit compared with average digit vectors")
plt.show()


